### 4. 상태
State는 에이전트가 현재 어떤 정보를 가지고 있는지를 표현하는 데이터 구조로, 파이썬에서는 TypedDict나 Pydantic BaseModel을 사용해 정의할 수 있습니다. TypedDict는 단순히 키와 값의 타입을 명시하는 수준이라 잘못된 타입이 들어가도 실행은 되지만, 타입 검사기(mypy 등)에서만 잡아줍니다. 반면 Pydantic BaseModel은 실제 실행 시점에 타입을 엄격하게 검사하여 잘못된 데이터가 들어오면 오류를 발생시킵니다. 따라서 State를 정의할 때는 상황에 따라 단순히 구조만 명시할 것인지, 혹은 런타임에서 데이터 유효성까지 보장할 것인지를 고려해 TypedDict와 Pydantic을 선택하여 활용합니다.

In [ ]:
from typing import TypedDict

class User(TypedDict):
    id: int
    name: str
    email: str

user1: User = {
    'id': 1,
    'name': '김사과',
    'email': 'apple@apple.com'
}
print(user1)

In [ ]:
user1: User = {
    'id': 1,
    'name': 12345678, # 숫자로 표현해도 잘 작동함.
    'email': 'apple@apple.com'
}
print(user1)

In [ ]:
from pydantic import BaseModel

class User(BaseModel):
    id: int
    name: str
    email: str

user_data = {
    'id': 1,
    'name': '김사과',
    'email': 'apple@apple.com'
}

user1 = User(**user_data) # key와 value가 일치하는 경우에만 작동함.
print(user1)

In [ ]:
user_data = {
    'id': 1,
    'name': 12345678,
    'email': 'apple@apple.com'
}

user = User(**user_data)    # pydantic은 타입 검사를 엄격하게 수행하기 때문에, 
                            # name 필드에 숫자가 들어가면 오류가 발생함.
                            # 오히려 오류가 발생하는 것이 더 안전한 코드라고 할 수 있음.
print(user)

# ValidationError: 1 validation error for User

State는 입력과 출력의 스키마를 정의하고 이를 업데이트하는 리듀서 함수와 함께 사용됩니다. LangGraph에서는 TypedDict 등을 이용해 입력과 출력 상태를 명확히 정의하고, 각 노드가 상태를 받아 새로운 값을 반환하면 그래프가 이를 이어받아 전체 흐름을 완성합니다. 즉, State는 질문과 답변 같은 에이전트의 맥락 정보를 구조적으로 관리하며, 노드 실행 결과를 반영해 에이전트의 상태를 단계적으로 업데이트하는 핵심 역할을 합니다.

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing_extensions import TypedDict


# 입력을 위한 스키마 정의
class InputState(TypedDict):
    question: str


# 출력을 위한 스키마 정의
class OutputState(TypedDict):
    answer: str


# 입력과 출력을 합한 종합 스키마 정의
class OverallState(InputState, OutputState):
    pass

In [ ]:
# 입력을 처리하고 답변을 생성하는 노드 정의
def answer_node(state: InputState):
    return {"answer": "bye", "question": state["question"]} # 상태 업데이트


graph_builder = StateGraph(OverallState, input_schema=InputState, output_schema=OutputState)
graph_builder.add_node(answer_node)  # 답변 노드 추가
graph_builder.add_edge(START, "answer_node")  # 시작 엣지 추가
graph_builder.add_edge("answer_node", END)  # 끝 엣지 추가
graph = graph_builder.compile()  # Compile the graph(그래프 준비해줘)

# 입력 invoke 및 결과 출력
print(graph.invoke({"question": "hi"}))

In [ ]:
graph

State는 에이전트가 사용하는 데이터 구조로, 단순히 값만 담는 것이 아니라 값이 어떻게 업데이트될지(리듀서 함수)까지 함께 정의할 수 있습니다. 기본 TypedDict를 사용하면 단순히 키와 값의 타입만 지정하지만, Annotated와 리듀서 함수를 함께 쓰면 상태 업데이트 시 단순 덮어쓰기 대신 지정된 동작(예: add로 리스트 이어 붙이기, add_messages로 대화 메시지 누적하기)을 수행합니다. 즉, State는 에이전트의 현재 정보를 표현할 뿐 아니라, 노드 실행 결과가 기존 상태와 어떻게 병합·갱신될지를 규칙으로 내장한 확장된 상태 관리 구조입니다.

In [ ]:
from typing_extensions import TypedDict

class State(TypedDict):
    value1: int
    value2: list[str]

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from operator import add

class State(TypedDict):
    value1: int
    # Annotated: 타입에 추가적인 메타데이터를 붙이는 방법
    # Annnotated[타입, 메타데이터]
    # add는 랭그래프에서 state를 합칠 때 리스트를 이어붙이는 규칙
    # like append
    value2: Annotated[list[str], add]

State = {
    "value1": 1,
    "value2": ["a", "b"]
}

State

In [ ]:
from typing_extensions import Annotated
from typing_extensions import TypedDict

def add(left, right):
    return left + right


class State(TypedDict):
    value1: int
    value2: Annotated[list[str], add]


State = {
    "value1": 1,
    "value2": ["a", "b"]
}

State

In [ ]:
from langchain_core.messages import AnyMessage
from langgraph.graph.message import add_messages
from typing import Annotated
from typing_extensions import TypedDict

class State(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]